In [3]:
import numpy as np
import pandas as pd
import datetime as dt
import scipy
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.cluster import k_means, KMeans
from scipy.cluster.hierarchy import dendrogram, linkage
import seaborn as sns

In [4]:
import sys

sys.path.append("../src")

In [5]:
from core import SITE_NAMES
from calls import plot_call_features, compute_features


random_state_for_sites = {'Foliage':800, 'Carp':200, 'Central':0, 'Telephone':0}
color_for_groups = {0: 'blue', 1: 'red', 2:'limegreen'}
label_for_groups = {0: 'LF1', 1: 'HF1', 2:'HF2'}

In [12]:
save_site = 'carp_and_telephone'
output_dir = Path(f'../data/generated_welch/{save_site}')
if not(output_dir.is_dir()):
    output_dir.mkdir(parents=True)
output_file_type = 'top1_inbouts_2ms_bandpass_welch_signals'

all_sites_welch_signals = []
for site_key in ['Carp', 'Telephone']:
    input_dir = Path(f'../data/detected_calls/{site_key}')
    input_file_type = 'top1_inbouts_2ms_bandpass_call_signals'
    if (input_dir / f'2022_bd2{site_key}_{input_file_type}.npy').exists():
        print(site_key)
        location_call_signals = np.load(input_dir / f'2022_bd2{site_key}_{input_file_type}.npy', allow_pickle=True)
        location_calls_sampled = pd.read_csv(input_dir / f'2022_bd2{site_key}_top1_inbouts_2ms_bandpass.csv', index_col=0, low_memory=False)
        location_calls_sampled['index_in_file'] = location_calls_sampled['index']
        location_calls_sampled['index'] = location_calls_sampled.index

        snr_thresh = 20
        good_snr_location_calls_sampled = location_calls_sampled.loc[location_calls_sampled['SNR']>=snr_thresh].copy()
        good_snr_location_calls_sampled.reset_index(drop=True, inplace=True)
        good_snr_location_calls_sampled

        welch_signals = compute_features.generate_welchs_for_calls(good_snr_location_calls_sampled, location_call_signals)
        all_sites_welch_signals.append(welch_signals)

all_sites_welch_signals = np.vstack(all_sites_welch_signals)
welch_data = pd.DataFrame(all_sites_welch_signals, columns=np.linspace(0, 96000, all_sites_welch_signals.shape[1]).astype(int))
welch_data.index.name = 'Call #'
welch_data.columns.name = 'Frequency (kHz)'
welch_data.to_csv(output_dir / f'2022_bd2{save_site}_{output_file_type}.csv')

Carp
Telephone
